#  HI fluxes

Here we aim to explore the HI fluxes from different catalogs using [edge_pydb](https://github.com/tonywong94/edge_pydb)

1. edge_leda (647 entries, but only 453 have fluxes)
2. edge_hiflux (161 entries; edge2015 and some more)
3. CALIFA_HI_sample_archive.csv (923 entries)

4. FASHI (41741 entries)
5. ALFALFA  (31502 entries) = a100

Also of use

1. iEDGE (643 entries) 
2. edge_califa (671 entries)
3. amusing (621 entries)

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from astropy.table import Table, join
from astropy import units as u
from edge_pydb import EdgeTable


In [ ]:
def common(t1, t2, c1, c2):
    print(len(t1),len(t2))
    t2c = t2[c2]
    mask = np.isin(t1[c1], t2c)
    return t1[mask]

In [ ]:
EdgeTable('list')

In [ ]:
# there's a "bug" in edge_pydb in that it doesn't recognize ecsv yet. I have a PR coming
if True:
    from edge_pydb import util
    util.updatefiles()
    EdgeTable('list')

In [ ]:
# note the comment line became a row
amusing = EdgeTable('amusing_sample_char.csv')
amusing.keys()

In [ ]:
#leda = pd.read_csv('edge_leda.csv')
#leda = np.loadtxt('edge_leda.csv', skiprows=1, delimiter=',')
#leda = np.genfromtxt('edge_leda.csv', skip_header=1, delimiter=',')
leda = EdgeTable('edge_leda.csv')

In [ ]:
leda.info()


In [ ]:
# Name,ledaHIflux
idx = ~leda['ledaHIflux'].mask

# 453 galaxies (out of 647) have a flux
leda_haveflux = leda['Name'][idx]
leda_flux = leda['ledaHIflux'][idx]
#leda_flux


In [ ]:
iedge = EdgeTable("iedge_v1.csv")

In [ ]:
iedge.info()

In [ ]:
edge_hiflux = EdgeTable("edge_hiflux.csv")


In [ ]:
edge_hiflux
# Name, SigInt, SigUnc
edge_hiflux['Name']


In [ ]:
califa = EdgeTable('edge_califa.csv')


In [ ]:
califa

## LEDA vs. EDGE

Take all edge_leda galaxies which have a known flux (no error bars available). Then find the flux in edge_hiflux for those galaxies

In [ ]:
#leda = EdgeTable('edge_leda.csv', cols=['Name', 'ledaHIflux'])
leda = EdgeTable('edge_leda.csv')

In [ ]:
edge_hiflux = EdgeTable("edge_hiflux.csv", cols=['Name', 'SigInt', 'SigUnc'])

In [ ]:
leda.join(edge_hiflux)
len(leda)

In [ ]:
leda.keys()
print(leda)

In [ ]:
fig, ax = plt.subplots()
ax.plot(leda['ledaHIflux'], leda['SigInt'], 'o')
ax.errorbar(leda['ledaHIflux'], leda['SigInt'], yerr=leda['SigUnc'],ls='none')
fmin = 0
fmax = 20
ax.set_xlim(fmin,fmax)
ax.set_ylim(fmin,fmax)
ax.grid()
ax.set_aspect('equal')
ax.plot([fmin,fmin],[fmax,fmax],'-', color='red')
ax.set_xlabel("LEDA flux")
ax.set_ylabel("EDGE_HIFLUX")

# Cross matching catalogs

```
 t1   edge_leda.csv
 t2   iedge_v1.ecsv
 t3   edge_hiflux.csv       only has names, no ra,dec
 t4   edge2025.csv          only has names, no ra,dec

In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord, Distance
from astropy.table import QTable
from astropy.table import Table, join
from astropy.time import Time


In [ ]:
#len(t1),len(t2),len(t3)

In [ ]:
t1 = QTable.read("edge_leda.csv", format="ecsv")
t1.keys()

In [ ]:
t1c = SkyCoord(t1["ledaRA"], t1["ledaDE"])
len(t1c)

In [ ]:
t1["ledaHIflux"]


In [ ]:
t2 = QTable.read("iedge_v1.ecsv", format="ecsv")
len(t2)
t2.keys()

In [ ]:
t2c = SkyCoord(t2["RA"], t2["DEC"])

In [ ]:
len(t2c)

In [ ]:
idx, sep, _ = t1c.match_to_catalog_sky(t2c)

In [ ]:
plt.hist(sep.arcsec, histtype="step", bins=np.logspace(-2, 2.0, 64))
plt.xlabel("separation [arcsec]")
plt.xscale("log")
plt.yscale("log")
plt.tight_layout()

In [ ]:
(sep < 2 * u.arcsec).sum(), len(t2)

In [ ]:
t3 = QTable.read("edge_hiflux.csv", format="ecsv")
t3n = t3["Name"]

In [ ]:
t31 = join(t3,t1)
len(t31)
t31.keys()

In [ ]:
flux1 = t31["ledaHIflux"]
flux3 = t31["SigInt"]


plt.plot(flux1,flux3,'o',label="data")
plt.plot([0,50], [0,50], '-', label="1:1")
plt.xlabel("LEDA")
plt.ylabel("GBT 2015+")
#plt.aspect("equal")
plt.legend();
plt.title(f"GBT + LEDA: {len(t31)} entries")

In [ ]:
t4 = Table.read("edge2025.csv", format="ecsv")
t4n = t4["Name"]

In [ ]:
t41 = join(t4,t1)
len(t41)

In [ ]:
flux1 = t41["ledaHIflux"]
flux4 = t41["SigInt"]

plt.plot(flux1,flux4,'o',label="data")
plt.plot([0,20], [0,20], '-', label="1:1")
plt.xlabel("LEDA")
plt.ylabel("GBT 2025")
plt.legend();
plt.title(f"GBT25 + LEDA: {len(t41)} entries")

In [ ]:
t41.keys()

In [ ]:
# panda's style doesn't work for tables
#  t2[t2["CALIFA_name"].isin(t3n)]

# would need to convert to pandas:
# pt2 = t2.to_pandas()


In [ ]:
# Create mask and extract rows
mask = np.isin(t2['CALIFA_name'], t3n)
t2_3 = t2[mask]
len(t2),len(t2_3)

In [ ]:
t1.keys()
t41 = common(t1,t4,"ledaName","Name")
t41

In [ ]:
a=common(t2,t3,"CALIFA_name","Name")
b=common(t3,t2,"Name","CALIFA_name")

In [ ]:
a

In [ ]:
b


In [ ]:
califa = QTable.read('edge_califa.csv', format="ecsv")
califa.keys()

In [ ]:
califa

In [ ]:
a100 = QTable.read('a100.code12.table2.190808.csv', format="csv")

In [ ]:
a100
